####**This was generated by Claude.ai on 28_07_2026, in chat 'Med2Vec-lite with CCSR for anomaly detection' at URL https://claude.ai/chat/3fdd808b-cf7d-4510-ac70-de3cf2f45dee**

The purpose was to import and process a data set from the file 'DXCCSR_v2026-1.csv', which was found at https://hcup-us.ahrq.gov/toolssoftware/ccsr/dxccsr.jsp.  That website is part of a project by AHRQ, the Agency for Healthcare Research and Quality (part of the US government Dept of Health & Human Services).  The goal was to collapse over 70,000 ICD-10-CM diagnosis codes into over 530 clinically meaningful categories.  These values can then be used in further machine learning and analysis of healthcare data (suchs as Med2Vec).

# CCSR for ICD-10-CM: load, clean, reshape, and distribution review

Working notebook for the MIHIN / Med2Vec-lite anomaly detection pipeline.
Covers:
1. Loading the raw AHRQ `DXCCSR_v2026-1.csv` (handles the file's mixed quoting convention)
2. Reshaping to a long (multi-label) `icd10 -> ccsr_cat` table
3. Category-size distribution review
4. Body-system distribution review

Set `CSV_PATH` below to wherever you have the file locally.

In [1]:
import pandas as pd

# CSV_PATH = "DXCCSR_v2026-1.csv"  # update to your local path




## 1. Load and clean

**Gotcha:** the AHRQ file mixes two quoting conventions. Codes and blank slots
are wrapped in *literal* single-quote characters that are just part of the
text (e.g. `'A067'`, `' '`), while descriptions containing commas use real
double-quote CSV escaping (e.g. `"Parasitic, other specified..."`).

So: parse with pandas' normal double-quote rules first, then strip stray `'`
characters off every field (including the header row) as a cleanup pass.
Don't set `quotechar="'"` — that breaks the double-quoted description fields.

In [ ]:
# Restored accidentally deleted cell:

dx_map = pd.read_csv(CSV_PATH, dtype=str)

def clean(s):
    if pd.isna(s):
        return s
    return s.strip().strip("'").strip()

dx_map.columns = [clean(c) for c in dx_map.columns]
for c in dx_map.columns:
    dx_map[c] = dx_map[c].map(clean)

# blank slots become '' after stripping -- normalize to NaN
dx_map = dx_map.replace('', pd.NA)

print("Shape:", dx_map.shape)
dx_map.head(3)

In [ ]:
print("Columns:")
for c in dx_map.columns:
    print(" -", c)

Columns:
 - ICD-10-CM CODE
 - ICD-10-CM CODE DESCRIPTION
 - Default CCSR CATEGORY IP
 - Default CCSR CATEGORY DESCRIPTION IP
 - Default CCSR CATEGORY OP
 - Default CCSR CATEGORY DESCRIPTION OP
 - CCSR CATEGORY 1
 - CCSR CATEGORY 1 DESCRIPTION
 - CCSR CATEGORY 2
 - CCSR CATEGORY 2 DESCRIPTION
 - CCSR CATEGORY 3
 - CCSR CATEGORY 3 DESCRIPTION
 - CCSR CATEGORY 4
 - CCSR CATEGORY 4 DESCRIPTION
 - CCSR CATEGORY 5
 - CCSR CATEGORY 5 DESCRIPTION
 - CCSR CATEGORY 6
 - CCSR CATEGORY 6 DESCRIPTION
 - Rationale for Default Assignment


## 2. Reshape to long format (multi-label mapping)

`CCSR CATEGORY 1` through `CCSR CATEGORY 6` hold the (up to 6) categories a
code can cross-classify into. Melt these into one row per (icd10, ccsr_cat)
pair, dropping empty slots.

In [ ]:
cat_cols = [c for c in dx_map.columns if c.startswith("CCSR CATEGORY") and "DESCRIPTION" not in c]

dx_long = (
    dx_map[["ICD-10-CM CODE"] + cat_cols]
    .rename(columns={"ICD-10-CM CODE": "icd10"})
    .melt(id_vars="icd10", value_name="ccsr_cat")
    .drop(columns="variable")
    .dropna(subset=["ccsr_cat"])
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Long table shape:", dx_long.shape)
print("Unique codes:", dx_long['icd10'].nunique())
print("Unique CCSR categories:", dx_long['ccsr_cat'].nunique())
dx_long.head()

Long table shape: (87705, 2)
Unique codes: 75725
Unique CCSR categories: 553


,icd10,ccsr_cat
0,A000,DIG001
1,A001,DIG001
2,A009,DIG001
3,A0100,DIG001
4,A0101,INF003


In [ ]:
# single-label default category (for the grouped-codes / mutually-exclusive use case)
dx_default = dx_map[["ICD-10-CM CODE", "Default CCSR CATEGORY IP", "Default CCSR CATEGORY OP"]].rename(
    columns={
        "ICD-10-CM CODE": "icd10",
        "Default CCSR CATEGORY IP": "ccsr_default_ip",
        "Default CCSR CATEGORY OP": "ccsr_default_op",
    }
)
dx_default.head()

,icd10,ccsr_default_ip,ccsr_default_op
0,A000,DIG001,DIG001
1,A001,DIG001,DIG001
2,A009,DIG001,DIG001
3,A0100,DIG001,DIG001
4,A0101,NVS001,NVS001


## 3. Sanity checks

Multi-mapping rate: what share of codes fall into more than one CCSR category.

In [ ]:
multi = dx_long.groupby("icd10").size()
print("Share of codes with >1 CCSR category:", round((multi > 1).mean(), 4))
print("Max categories on a single code:", multi.max())

Share of codes with >1 CCSR category: 0.1183
Max categories on a single code: 6


## 4. Category-size distribution

How many ICD-10-CM codes map into each of the 553 CCSR categories.

In [ ]:
cat_counts = dx_long.groupby("ccsr_cat").size().sort_values(ascending=False)
print(cat_counts.describe())

count     553.000000
mean      158.598553
std       487.222896
min         1.000000
25%        16.000000
50%        48.000000
75%       123.000000
max      7655.000000
dtype: float64


In [ ]:
# description lookup, built from the wide table's per-slot description columns
desc_lookup = {}
for i in range(1, 7):
    code_col = f"CCSR CATEGORY {i}"
    desc_col = f"CCSR CATEGORY {i} DESCRIPTION"
    sub = dx_map[[code_col, desc_col]].dropna()
    for c, d in zip(sub[code_col], sub[desc_col]):
        desc_lookup[c] = d

print("Top 10 largest categories:")
for cat, n in cat_counts.head(10).items():
    print(f"  {cat:8s} {n:5d}  {desc_lookup.get(cat, '')}")

print("\nBottom 10 smallest categories:")
for cat, n in cat_counts.tail(10).items():
    print(f"  {cat:8s} {n:5d}  {desc_lookup.get(cat, '')}")

print("\nCategories with only 1 code:", (cat_counts == 1).sum())

Top 10 largest categories:
  INJ073    7655  Injury, sequela
  INJ042    4702  Fracture of lower limb (except hip), subsequent encounter
  INJ041    4348  Fracture of the upper limb, subsequent encounter
  EXT029    2415  External cause codes: subsequent encounter
  EXT030    2415  External cause codes: sequela
  EXT020    2403  External cause codes: intent of injury, accidental/unintentional
  INJ075    1621  Poisoning/toxic effect/adverse effects/underdosing, sequela
  INJ004    1555  Fracture of the upper limb, initial encounter
  INJ005    1435  Fracture of the lower limb (except hip), initial encounter
  INJ019     936  Burn and corrosion, initial encounter

Bottom 10 smallest categories:
  NEO037       2  Female reproductive system cancers - vagina
  NEO059       2  Leukemia - acute lymphoblastic leukemia (ALL)
  NEO019       1  Gastrointestinal cancers - gallbladder
  NEO018       1  Gastrointestinal cancers - bile duct
  NEO052       1  Endocrine system cancers - thymus
  NEO05

## 5. Body-system distribution

Body system = first 3 letters of the CCSR category code (e.g. `INJ`, `CIR`, `NEO`).

In [ ]:
dx_long["body_system"] = dx_long["ccsr_cat"].str[:3]

sys_summary = dx_long.groupby("body_system").agg(
    n_categories=("ccsr_cat", "nunique"),
    n_codes=("icd10", "nunique"),
).sort_values("n_codes", ascending=False)

sys_summary

,n_categories,n_codes
body_system,,
INJ,76,41865
EXT,30,9190
MUS,38,6752
EYE,12,3061
PRG,30,2565
MBD,32,2269
NEO,78,1829
CIR,39,1776
INF,12,1330


**Note:** this is a code-*count* distribution, not a visit-*frequency*
distribution. `INJ` (injury and poisoning) dominates because ICD-10-CM's
injury coding is combinatorial (body part x injury type x laterality x
encounter type), not necessarily because injury visits are actually more
common in your data. Join `dx_long` against real visit data before assuming
this skew carries through to training.

In [ ]:
# cache cleaned tables for downstream notebooks
dx_map.to_parquet("dx_map_clean.parquet")
dx_long.to_parquet("dx_long.parquet")
print("saved dx_map_clean.parquet, dx_long.parquet")

saved dx_map_clean.parquet, dx_long.parquet
